In [1]:
import os
timeline_directory = "/Volumes/cnlab/GeoRemote/Timeline/2_S2"
output_file_path = "/Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/"
matching_timeline_folders = sorted([folder for folder in os.listdir(timeline_directory) if os.path.isdir(os.path.join(timeline_directory, folder))],reverse=True)
output_files = sorted([folder for folder in os.listdir(output_file_path)],reverse=True)

In [2]:
import re
import json
import csv
import ijson
import pandas as pd


def find_file(start_dir, file_name):
    for root, dirs, files in os.walk(start_dir):
        if file_name in files:
            return os.path.join(root, file_name)

    # File not found
    return None

def extract_participant_id(input_string):
    match = re.search(r'(GR\d{3})', input_string)
    if match:
        participant_id = match.group(1)
        return participant_id
    else:
        return None
    
def get_first_device_tag(records_file):
    with open(records_file, 'r') as file:
        objects = ijson.items(file, 'locations.item')
        for obj in objects:
            if 'deviceTag' in obj:
                return obj['deviceTag']
    return None

def find_device_settings(settings_file, device_tag):
    with open(settings_file, 'r') as file:
        settings = json.load(file)
        for device in settings['deviceSettings']:
            if device['deviceTag'] == device_tag:
                device_spec = device.get('deviceSpec', {})
                return {
                    'deviceTag': device.get('deviceTag'),
                    'reportingEnabled': device.get('reportingEnabled'),
                    'devicePrettyName': device.get('devicePrettyName'),
                    'platformType': device.get('platformType'),
                    'deviceCreationTime': device.get('deviceCreationTime'),
                    'reportingEnabledModificationTime': device.get('latestLocationReportingSettingChange', {}).get('reportingEnabledModificationTime'),
                    'androidOsLevel': device.get('androidOsLevel'),
                    'iosVersion': device.get('iosVersion'),
                    'manufacturer': device_spec.get('manufacturer'),
                    'brand': device_spec.get('brand'),
                    'product': device_spec.get('product'),
                    'device': device_spec.get('device'),
                    'model': device_spec.get('model'),
                    'isLowRam': device_spec.get('isLowRam')
                }
    return None

def write_device_settings_to_csv(device_info, output_csv_file):
    # Define the CSV headers
    headers = [
        'deviceTag', 'reportingEnabled', 'devicePrettyName', 'platformType', 
        'deviceCreationTime', 'reportingEnabledModificationTime', 'androidOsLevel', 'iosVersion',
        'manufacturer', 'brand', 'product', 'device', 'model', 'isLowRam'
    ]
    
    # Create a DataFrame from the device_info dictionary
    df = pd.DataFrame([device_info], columns=headers)
    
    # Replace None with NA
    df = df.where(pd.notnull(df), 'NA')
    
    # Write the DataFrame to a CSV file
    df.to_csv(output_csv_file, index=False)



In [5]:
for timeline_folder in matching_timeline_folders:
    ppt = extract_participant_id(timeline_folder)
    output_file = os.path.join(output_file_path, f"{ppt}_descriptives.csv")
    if ppt is not None:
        if any(ppt in s for s in output_files):
            continue
        settings_file = find_file(os.path.join(timeline_directory,timeline_folder), "Settings.json")
        records_file = find_file("/Volumes/cnlab/GeoRemote/Data/Geodata/clean/trimmed_geodata/", f"{ppt}_geodata.json")
        if records_file:
            device_tag = get_first_device_tag(records_file)
            if device_tag:
                device_info = find_device_settings(settings_file, device_tag)
                if device_info:
                    write_device_settings_to_csv(device_info, output_file)
                    print(f"Device settings for {ppt} have been written to {output_file}")
                else:
                    print(f"No device settings found for deviceTag {device_tag}")
            else:
                print("No deviceTag found in the Records.json")



Device settings for GR319 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR319_descriptives.csv
Device settings for GR318 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR318_descriptives.csv
Device settings for GR317 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR317_descriptives.csv
Device settings for GR316 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR316_descriptives.csv
Device settings for GR315 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR315_descriptives.csv
Device settings for GR313 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR313_descriptives.csv
Device settings for GR312 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR312_descriptives.csv
Device settings for GR310 have been written to /Volumes/cnlab/GeoRemote/Data/Geodata/clean/descriptives/GR310_d

KeyboardInterrupt: 